# Implementation Figures: Slides 6 To 10

This notebook builds PowerPoint-ready static assets for the remaining implementation-phase slides. It uses the cleaned DREF-family datasets from the shared prep notebook and selectively brings `EA` context back in where the slide narrative explicitly requires it.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', 120)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

In [2]:
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

WORKBOOK_PATH = ROOT / 'DREF_MasterDataset_v1.1 .xlsx'
PROCESSED_DIR = ROOT / 'data' / 'processed'
OUTPUT_DIR = ROOT / 'outputs' / 'implementation_phase_python'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

q1_history = pd.read_csv(PROCESSED_DIR / 'q1_history_base_dref.csv', parse_dates=['approval_date', 'date_of_disaster_trigger_date', 'date_of_appeal_request_from_ns'])
q1_2026 = pd.read_csv(PROCESSED_DIR / 'q1_2026_base_dref.csv', parse_dates=['approval_date', 'date_of_disaster_trigger_date', 'date_of_appeal_request_from_ns'])
oda_countries = set(pd.read_csv(PROCESSED_DIR / 'oda_countries_reference.csv')['country'].dropna().astype(str).str.strip())

def slugify_column(name: str) -> str:
    text = re.sub(r'[^0-9a-zA-Z]+', '_', str(name).strip().lower())
    text = re.sub(r'_+', '_', text).strip('_')
    return text

all_data_raw = pd.read_excel(WORKBOOK_PATH, sheet_name='ALL_DATA')
all_data = all_data_raw.rename(columns={column: slugify_column(column) for column in all_data_raw.columns}).copy()
for column in ['appeal_type', 'country', 'region', 'pillar', 'allocation_type', 'disaster_definition']:
    all_data[column] = all_data[column].fillna('').astype(str).str.strip()
for column in ['date_of_appeal_request_from_ns', 'date_of_approval_enc_start_date']:
    all_data[column] = pd.to_datetime(all_data[column], errors='coerce')
all_data['approval_date'] = all_data['date_of_approval_enc_start_date']
all_data['approval_year'] = all_data['approval_date'].dt.year
all_data['total_approved_chf'] = pd.to_numeric(all_data['total_approved_chf'], errors='coerce').fillna(0)

# Flourish-inspired style system (shared with all notebooks)
FLR_PAPER = '#fafafa'
FLR_PLOT = '#fafafa'
FLR_GRID = '#e8e8e8'
FLR_LINE = '#d0d0d0'

FLR_FONT = 'Inter, Segoe UI, Helvetica Neue, Arial, sans-serif'
FLR_TITLE = {'size': 30, 'color': '#1a1a2e', 'family': f'Inter, Segoe UI Semibold, {FLR_FONT}'}
FLR_SUBTITLE = {'size': 18, 'color': '#666666', 'family': FLR_FONT}
FLR_AXIS_LBL = {'size': 20, 'color': '#333333', 'family': FLR_FONT}
FLR_TICK = {'size': 17, 'color': '#555555', 'family': FLR_FONT}

IFRC_RED = '#E03C31'
NAVY = '#2B3A67'
AMBER = '#F5A623'
TEAL = '#2EC4B6'
GREEN = '#4f8a10'
PURPLE = '#9B72CF'
GREY = '#D5D5D5'
INK = '#222222'
BG = '#fafafa'

REGION_COLORS = {
    'Africa': '#E8733A',
    'Americas': '#3DAD6E',
    'Asia-Pacific': '#4ECDC4',
    'Europe': '#9B72CF',
    'MENA': '#4A90D9',
    'Global': '#B0B0B0',
}

HAZARD_PALETTE = ['#E03C31', '#F08A4B', '#F5C518', '#2EC4B6', '#4A90D9']


def flr_legend(y=-0.22, x=0.5):
    return dict(
        orientation='h',
        xanchor='center',
        x=x,
        yanchor='top',
        y=y,
        bgcolor='rgba(0,0,0,0)',
        borderwidth=0,
        font={'size': 15, 'color': '#555555', 'family': FLR_FONT},
        itemsizing='constant',
    )


def flr_xaxis(title='', **overrides):
    base = dict(
        title={'text': title, 'font': FLR_AXIS_LBL, 'standoff': 16},
        tickfont=FLR_TICK,
        showgrid=False,
        zeroline=False,
        showline=True,
        linecolor=FLR_LINE,
        linewidth=1,
        tickcolor=FLR_LINE,
        ticks='outside',
        ticklen=6,
        tickwidth=1,
        automargin=True,
    )
    base.update(overrides)
    return base


def flr_yaxis(title='', **overrides):
    base = dict(
        title={'text': title, 'font': FLR_AXIS_LBL, 'standoff': 16},
        tickfont=FLR_TICK,
        gridcolor=FLR_GRID,
        gridwidth=0.7,
        showgrid=True,
        zeroline=False,
        showline=False,
        ticks='outside',
        ticklen=6,
        tickwidth=1,
        automargin=True,
    )
    base.update(overrides)
    return base


def flr_layout(title_text, subtitle='', height=700, margin_l=90, margin_r=50, margin_t=125, margin_b=150):
    title_html = title_text
    if subtitle:
        title_html += f'<br><span style="font-size:18px;color:#888;">{subtitle}</span>'
    return dict(
        title={
            'text': title_html,
            'x': 0.5,
            'xanchor': 'center',
            'font': FLR_TITLE,
            'pad': {'b': 18},
        },
        paper_bgcolor=FLR_PAPER,
        plot_bgcolor=FLR_PLOT,
        margin={'l': margin_l, 'r': margin_r, 't': margin_t, 'b': margin_b},
        height=height,
        font={'family': FLR_FONT, 'size': 15, 'color': '#444444'},
        hoverlabel=dict(
            bgcolor='white',
            bordercolor='#ddd',
            font_size=14,
            font_family=FLR_FONT,
        ),
    )


def human_chf(value: float) -> str:
    if pd.isna(value):
        return 'n/a'
    if abs(value) >= 1_000_000:
        return f'CHF {value / 1_000_000:.1f}M'
    if abs(value) >= 1_000:
        return f'CHF {value / 1_000:.0f}K'
    return f'CHF {value:,.0f}'


def save_figure(fig: go.Figure, stem: str, width: int = 1400, height: int = 900) -> None:
    fig.write_image(OUTPUT_DIR / f'{stem}.png', width=width, height=height, scale=2)
    fig.write_image(OUTPUT_DIR / f'{stem}.svg', width=width, height=height)

print('Workbook:', WORKBOOK_PATH)
print('Workbook exists:', WORKBOOK_PATH.exists())
print('Processed dir:', PROCESSED_DIR)
print('Output dir:', OUTPUT_DIR)

Workbook: C:\Users\arun.gandhi\Downloads\DREF_GA_visualizations\DREF_MasterDataset_v1.1 .xlsx
Workbook exists: True
Processed dir: C:\Users\arun.gandhi\Downloads\DREF_GA_visualizations\data\processed
Output dir: C:\Users\arun.gandhi\Downloads\DREF_GA_visualizations\outputs\implementation_phase_python


c:\Users\arun.gandhi\Downloads\DREF_GA_visualizations\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


## Slide 6

This version uses two parallel composition charts rather than a four-segment combination, because the cross-classification in the data is not rich enough to justify a more complex construct.

In [28]:
# Slide 6: Hazard Composition
slide6 = q1_history[q1_history['approval_year'].between(2022, 2026)].copy()
years = [2022, 2023, 2024, 2025, 2026]

weather_categories = ['Weather-related', 'Non-weather-related', 'Other']
natural_categories = ['Natural', 'Non-natural', 'Other']
weather_colors = {'Weather-related': IFRC_RED, 'Non-weather-related': NAVY, 'Other': '#D9D9D9'}
natural_colors = {'Natural': IFRC_RED, 'Non-natural': NAVY, 'Other': '#D9D9D9'}


def build_mix(source: pd.DataFrame, category_col: str, categories: list[str]) -> tuple[pd.DataFrame, pd.Series]:
    values = source.groupby(['approval_year', category_col], as_index=False)['total_approved_chf'].sum()
    pivot = (
        values.pivot(index='approval_year', columns=category_col, values='total_approved_chf')
        .reindex(index=years, columns=categories, fill_value=0)
        .fillna(0)
    )
    totals = pivot.sum(axis=1)
    shares = pivot.div(totals.replace(0, np.nan), axis=0).fillna(0)
    return shares, totals


def add_panel_key(fig: go.Figure, items: list[tuple[str, str]], x_start: float, y: float, step: float = 0.12) -> None:
    for index, (label, color) in enumerate(items):
        x0 = x_start + index * step
        fig.add_shape(
            type='rect',
            xref='paper',
            yref='paper',
            x0=x0,
            x1=x0 + 0.012,
            y0=y - 0.012,
            y1=y + 0.012,
            fillcolor=color,
            line={'width': 0},
        )
        fig.add_annotation(
            x=x0 + 0.016,
            y=y,
            xref='paper',
            yref='paper',
            text=label,
            showarrow=False,
            xanchor='left',
            yanchor='middle',
            font={'size': 12, 'color': '#555555', 'family': FLR_FONT},
        )


weather_share, weather_totals = build_mix(slide6, 'weather_vs_non_weather', weather_categories)
natural_share, natural_totals = build_mix(slide6, 'natural_vs_non_natural', natural_categories)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=['Weather Classification', 'Natural Classification'],
    horizontal_spacing=0.16,
)

for category in weather_categories:
    values = weather_share[category]
    fig.add_trace(
        go.Bar(
            x=years,
            y=values,
            name=category,
            marker_color=weather_colors[category],
            marker_line_width=0,
            text=[f'{value:.0%}' if value >= 0.10 else '' for value in values],
            textposition='inside',
            textfont={'size': 14, 'color': 'white', 'family': FLR_FONT},
            hovertemplate=f'{category}<br>%{{x}}: %{{y:.1%}}<extra></extra>',
            showlegend=False,
        ),
        row=1,
        col=1,
    )

for category in natural_categories:
    values = natural_share[category]
    fig.add_trace(
        go.Bar(
            x=years,
            y=values,
            name=category,
            marker_color=natural_colors[category],
            marker_line_width=0,
            text=[f'{value:.0%}' if value >= 0.10 else '' for value in values],
            textposition='inside',
            textfont={'size': 14, 'color': 'white', 'family': FLR_FONT},
            hovertemplate=f'{category}<br>%{{x}}: %{{y:.1%}}<extra></extra>',
            showlegend=False,
        ),
        row=1,
        col=2,
    )

for year in years:
    fig.add_annotation(
        x=year,
        y=1.035,
        xref='x',
        yref='y',
        text=f'<span style="font-size:11px;color:#888;">{human_chf(weather_totals.loc[year])}</span>',
        showarrow=False,
    )
    fig.add_annotation(
        x=year,
        y=1.035,
        xref='x2',
        yref='y2',
        text=f'<span style="font-size:11px;color:#888;">{human_chf(natural_totals.loc[year])}</span>',
        showarrow=False,
    )

weather_2026 = weather_share.loc[2026, 'Weather-related']
natural_2026 = natural_share.loc[2026, 'Natural']

# Bottom annotations — "Q1 approval year" shared x-axis label + insight caption
fig.add_annotation(
    x=0.5,
    y=-0.08,
    xref='paper',
    yref='paper',
    text='Q1 approval year',
    showarrow=False,
    font={'size': 16, 'color': '#444444', 'family': FLR_FONT},
)
fig.add_annotation(
    x=0.5,
    y=-0.16,
    xref='paper',
    yref='paper',
    text=(
        f'2025 was the only non-weather / non-natural Q1 in the series; '
        f'2026 returns to <b>{weather_2026:.0%}</b> weather-related and '
        f'<b>{natural_2026:.0%}</b> natural allocations.'
    ),
    showarrow=False,
    font={'size': 14, 'color': '#666666', 'family': FLR_FONT},
)

fig.update_layout(
    barmode='stack',
    **flr_layout(
        'Hazard Composition',
        subtitle='Share of approved CHF by classification — Q1 2022 to 2026',
        height=800,
        margin_l=90,
        margin_r=60,
        margin_t=200,   # keeps subtitle ~30 px above panel keys (y=1.10)
        margin_b=120,
    ),
)

fig.update_yaxes(
    **flr_yaxis('Share of approved CHF', range=[0, 1.08], tickformat='.0%'),
    row=1,
    col=1,
)
fig.update_yaxes(
    **flr_yaxis('Share of approved CHF', range=[0, 1.08], tickformat='.0%'),
    row=1,
    col=2,
)
fig.update_xaxes(**flr_xaxis('', tickmode='array', tickvals=years), row=1, col=1)
fig.update_xaxes(**flr_xaxis('', tickmode='array', tickvals=years), row=1, col=2)

for ann in fig.layout.annotations:
    if ann.text in ['Weather Classification', 'Natural Classification']:
        ann.update(font={'size': 18, 'color': '#444444', 'family': FLR_FONT})

# Panel keys at y=1.10 — comfortably below the subtitle (~y≈1.16) and
# above the subplot titles (~y≈1.05), with ~30 px clearance on each side.
add_panel_key(
    fig,
    [('Weather-related', IFRC_RED), ('Non-weather-related', NAVY), ('Other', '#D9D9D9')],
    x_start=0.11,
    y=1.10,
    step=0.15,
)
add_panel_key(
    fig,
    [('Natural', IFRC_RED), ('Non-natural', NAVY), ('Other', '#D9D9D9')],
    x_start=0.61,
    y=1.10,
    step=0.13,
)

save_figure(fig, 'slide_06_hazard_composition_small_multiples', width=1500, height=800)
fig


## Slide 7

This version uses like-for-like Q1 comparisons across years rather than mixing full-year values with Q1 2026. The non-ODA flag is derived from the workbook reference sheet by exclusion.

In [9]:
# Slide 7: Non-ODA Allocations
slide7 = q1_history[q1_history['approval_year'].between(2023, 2026)].copy()
slide7['is_non_oda'] = ~slide7['country'].isin(oda_countries)
years = [2023, 2024, 2025, 2026]

year_totals = slide7.groupby('approval_year', as_index=False).agg(all_total=('total_approved_chf', 'sum'))
non_oda = (
    slide7[slide7['is_non_oda']]
    .groupby('approval_year', as_index=False)
    .agg(total_chf=('total_approved_chf', 'sum'), allocations=('appeal_id', 'size'))
)
non_oda = (
    pd.DataFrame({'approval_year': years})
    .merge(year_totals, on='approval_year', how='left')
    .merge(non_oda, on='approval_year', how='left')
    .fillna({'total_chf': 0, 'allocations': 0})
)
non_oda['allocations'] = non_oda['allocations'].astype(int)
non_oda['oda_total'] = non_oda['all_total'] - non_oda['total_chf']
non_oda['share'] = (non_oda['total_chf'] / non_oda['all_total'].replace(0, np.nan)).fillna(0)

non_oda_2026_cases = (
    slide7[(slide7['approval_year'] == 2026) & (slide7['is_non_oda'])]['country']
    .drop_duplicates()
    .sort_values()
    .tolist()
)
if non_oda_2026_cases:
    first_line = ', '.join(non_oda_2026_cases[:3])
    second_line = ', '.join(non_oda_2026_cases[3:])
    cases_text = first_line if not second_line else first_line + '<br>' + second_line
else:
    cases_text = 'No Non-ODA cases recorded'

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.62, 0.38], vertical_spacing=0.08)

fig.add_trace(
    go.Bar(
        x=years,
        y=non_oda['total_chf'],
        name='Non-ODA',
        marker_color=IFRC_RED,
        marker_line_width=0,
        hovertemplate='Non-ODA CHF<br>%{x}: CHF %{y:,.0f}<extra></extra>',
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Bar(
        x=years,
        y=non_oda['oda_total'],
        name='ODA / other Q1 allocations',
        marker_color='#DCE3EC',
        marker_line_color='#C6D0DB',
        marker_line_width=1,
        hovertemplate='ODA / other Q1 CHF<br>%{x}: CHF %{y:,.0f}<extra></extra>',
    ),
    row=1,
    col=1,
)

for row in non_oda.itertuples(index=False):
    fig.add_annotation(
        x=row.approval_year,
        y=row.total_chf,
        xref='x',
        yref='y',
        text=f'<b>{human_chf(row.total_chf)}</b>',
        showarrow=False,
        yshift=12,
        font={'size': 14, 'color': INK, 'family': FLR_FONT},
    )
    fig.add_annotation(
        x=row.approval_year,
        y=row.all_total,
        xref='x',
        yref='y',
        text=f'Total {human_chf(row.all_total)}',
        showarrow=False,
        yshift=14,
        font={'size': 11, 'color': '#98A2AE', 'family': FLR_FONT},
    )

fig.add_trace(
    go.Scatter(
        x=years,
        y=non_oda['share'],
        mode='lines+markers+text',
        line={'color': NAVY, 'width': 3},
        marker={'size': 12, 'color': NAVY, 'line': {'color': FLR_PAPER, 'width': 1.5}},
        text=[f'{share:.1%}' for share in non_oda['share']],
        textposition='top center',
        textfont={'size': 14, 'color': NAVY, 'family': FLR_FONT},
        showlegend=False,
        hovertemplate='Share of total<br>%{x}: %{y:.1%}<extra></extra>',
    ),
    row=2,
    col=1,
)

fig.add_annotation(
    x=0.97,
    y=0.33,
    xref='paper',
    yref='paper',
    text=f'<b>2026 cases</b><br>{cases_text}',
    showarrow=False,
    xanchor='right',
    bgcolor='white',
    bordercolor=IFRC_RED,
    borderwidth=1.5,
    borderpad=8,
    font={'size': 12, 'color': INK, 'family': FLR_FONT},
)
fig.add_annotation(
    x=0.5,
    y=-0.15,
    xref='paper',
    yref='paper',
    text='2026 rebounds from the 2025 low, but Non-ODA share remains below the 2023–2024 level',
    showarrow=False,
    font={'size': 14, 'color': '#666666', 'family': FLR_FONT},
)

fig.update_layout(
    barmode='stack',
    **flr_layout(
        'Non-ODA Allocations (Q1)',
        subtitle='Non-ODA value and share of total Q1 CHF — 2023 to 2026',
        height=860,
        margin_l=90,
        margin_r=90,
        margin_t=145,
        margin_b=140,
    ),
    legend=flr_legend(y=1.02),
)
fig.update_yaxes(**flr_yaxis('Approved CHF'), row=1, col=1)
fig.update_yaxes(
    **flr_yaxis('Share of total', tickformat='.0%', range=[0, max(0.14, non_oda['share'].max() + 0.025)]),
    row=2,
    col=1,
)
fig.update_xaxes(**flr_xaxis('Q1 approval year', tickmode='array', tickvals=years), row=2, col=1)
fig.update_xaxes(**flr_xaxis('', tickmode='array', tickvals=years), row=1, col=1)

save_figure(fig, 'slide_07_non_oda_q1_like_for_like', width=1400, height=860)
fig

## Slide 8

The ranked hazard figure is a strong fit for PowerPoint. This version uses a stacked horizontal bar so the regional composition remains attached to each hazard instead of being pushed into bullet text.

In [12]:
# Slide 8: Regional Hazard Mix in 2026 Q1
slide8 = q1_2026[q1_2026['region'].isin(['Africa', 'Americas', 'Asia-Pacific', 'Europe', 'MENA'])].copy()
hazard_totals = (
    slide8[slide8['disaster_definition'] != 'Other']
    .groupby('disaster_definition', as_index=False)
    .agg(total_chf=('total_approved_chf', 'sum'))
    .sort_values('total_chf', ascending=False)
)
focus_hazards = hazard_totals.head(5)['disaster_definition'].tolist()
hazard_order = focus_hazards + ['Other hazards']
hazard_colors = {
    'Flood': IFRC_RED,
    'Epidemic': '#F08A4B',
    'Population Movement': '#F5C518',
    'Complex Emergency': TEAL,
    'Fire': '#4A90D9',
    'Other hazards': '#D5D5D5',
}

slide8['hazard_group'] = np.where(
    slide8['disaster_definition'].isin(focus_hazards),
    slide8['disaster_definition'],
    'Other hazards',
)

region_hazard = (
    slide8.groupby(['region', 'hazard_group'], as_index=False)
    .agg(total_chf=('total_approved_chf', 'sum'))
)
hazard_matrix = (
    region_hazard.pivot(index='region', columns='hazard_group', values='total_chf')
    .reindex(columns=hazard_order, fill_value=0)
    .fillna(0)
)
region_totals = (
    slide8.groupby('region', as_index=False)
    .agg(total_chf=('total_approved_chf', 'sum'), ops=('appeal_id', 'nunique'))
    .sort_values('total_chf', ascending=False)
)
region_order = region_totals['region'].tolist()
hazard_matrix = hazard_matrix.reindex(index=region_order, fill_value=0)

fig = go.Figure()
for hazard in hazard_order:
    values = hazard_matrix[hazard]
    fig.add_trace(
        go.Bar(
            x=values,
            y=hazard_matrix.index,
            orientation='h',
            name=hazard,
            marker_color=hazard_colors[hazard],
            marker_line_width=0,
            hovertemplate=f'{hazard}<br>%{{y}}: CHF %{{x:,.0f}}<extra></extra>',
        )
    )

for row in region_totals.itertuples(index=False):
    fig.add_annotation(
        x=row.total_chf,
        y=row.region,
        text=f'<b>{human_chf(row.total_chf)}</b> │ {row.ops} ops',
        showarrow=False,
        xanchor='left',
        xshift=12,
        font={'size': 14, 'color': INK, 'family': FLR_FONT},
    )

fig.add_annotation(
    x=0.5,
    y=-0.16,
    xref='paper',
    yref='paper',
    text='Flood dominates Africa and MENA, while Europe is led by population movement and Asia-Pacific by complex emergency',
    showarrow=False,
    font={'size': 14, 'color': '#666666', 'family': FLR_FONT},
)

fig.update_layout(
    barmode='stack',
    **flr_layout(
        'Regional Hazard Mix in 2026 Q1',
        subtitle='Five regions on the y-axis, with approved CHF split across the five largest named hazards plus remainder',
        height=620,
        margin_l=160,
        margin_r=200,
        margin_t=110,
        margin_b=150,
    ),
    legend=flr_legend(y=-0.14),
    legend_traceorder='normal',
)
fig.update_xaxes(
    **flr_xaxis('', range=[0, region_totals['total_chf'].max() * 1.22]),
)
fig.update_yaxes(
    **flr_yaxis('', showgrid=False, categoryorder='array', categoryarray=region_order[::-1], tickfont={'size': 16, 'color': INK, 'family': FLR_FONT}),
)

save_figure(fig, 'slide_08_top_hazards_ranked', width=1500, height=760)
fig

## Slide 9

This figure keeps the timing trend but adds a framing note for 2026 `Days in HQ`, where the mean is influenced by outliers even though the median stays low.

In [16]:
# ── Slide 9: Timeliness ───────────────────────────────────────────────────────
slide9 = q1_history[q1_history['approval_year'].between(2023, 2026)].copy()
slide9['disaster_to_approval_days'] = slide9['average_time_disaster_to_approval']
slide9['ns_to_approval_days'] = (slide9['approval_date'] - slide9['date_of_appeal_request_from_ns']).dt.days
slide9['days_in_hq'] = pd.to_numeric(slide9['days_in_hq'], errors='coerce')

metrics = []
for label, column in [('Disaster to Approval', 'disaster_to_approval_days'),
                       ('NS Request to Approval', 'ns_to_approval_days'),
                       ('Days in HQ', 'days_in_hq')]:
    grouped = slide9.groupby('approval_year')[column].agg(['mean', 'median']).reset_index()
    grouped['metric'] = label
    metrics.append(grouped)
slide9_metrics = pd.concat(metrics, ignore_index=True)

fig = go.Figure()

# 7-day target zone — opacity raised to 0.30 for clear visibility
fig.add_shape(
    type='rect', x0=2022.6, x1=2026.4, y0=0, y1=7,
    fillcolor='rgba(46, 196, 182, 0.30)', line={'width': 0}, layer='below'
)
fig.add_annotation(
    x=2022.7, y=7, xref='x', yref='y',
    text='7-day target zone',
    showarrow=False, yshift=10, xanchor='left',
    font={'size': 11, 'color': '#2EC4B6', 'family': FLR_FONT},
)

metric_colors = {
    'Disaster to Approval': IFRC_RED,
    'NS Request to Approval': NAVY,
    'Days in HQ': AMBER,
}
for metric, color in metric_colors.items():
    subset = slide9_metrics[slide9_metrics['metric'] == metric].sort_values('approval_year')
    fig.add_trace(go.Scatter(
        x=subset['approval_year'],
        y=subset['mean'],
        mode='lines+markers+text',
        name=metric,
        line={'color': color, 'width': 2.5, 'shape': 'spline'},
        marker={'size': 10, 'color': color, 'line': {'color': FLR_PAPER, 'width': 1.5}},
        text=[f'{v:.1f}' for v in subset['mean']],
        textposition='top center',
        textfont={'color': color, 'size': 12, 'family': FLR_FONT},
    ))

fig.update_layout(
    **flr_layout(
        'Timeliness of Approvals (Q1)',
        subtitle='Mean processing days — 2025 spike reversed in 2026',
        height=560,
        margin_l=70,
        margin_r=60,
        margin_t=95,
        margin_b=90,
    ),
    legend=flr_legend(y=-0.15),
    xaxis=flr_xaxis('Q1 approval year', tickmode='linear', dtick=1),
    yaxis=flr_yaxis('Days'),
)

save_figure(fig, 'slide_09_timeliness_q1', width=1400, height=560)
fig


## Slide 10

This slide intentionally mixes `EA` and `i-DREF` because the narrative is about a cross-instrument emergency response sequence. The figure is exported as a timeline that can be paired with narrative bullets in PowerPoint.

In [10]:
# Slide 10: Cross-Regional Response Sequence
crisis_start = pd.Timestamp('2026-02-28')
countries = ['Iran', 'Lebanon', 'Armenia', 'Iraq', 'Azerbaijan', 'Turkmenistan']
slide10 = all_data[
    all_data['country'].isin(countries)
    & all_data['approval_year'].eq(2026)
    & all_data['appeal_type'].isin(['EA', 'i-DREF'])
].copy()
slide10 = slide10[[
    'country', 'region', 'appeal_type', 'allocation_type', 'disaster_definition',
    'total_approved_chf', 'date_of_appeal_request_from_ns', 'approval_date'
]].dropna(subset=['date_of_appeal_request_from_ns', 'approval_date'])
slide10['duration_days'] = (slide10['approval_date'] - slide10['date_of_appeal_request_from_ns']).dt.days
slide10['approval_display'] = slide10.apply(
    lambda row: row['approval_date'] + pd.Timedelta(hours=18) if row['duration_days'] == 0 else row['approval_date'],
    axis=1,
)
slide10['request_lag_days'] = (slide10['date_of_appeal_request_from_ns'] - crisis_start).dt.days
slide10['row_label'] = slide10['country'] + ' (' + slide10['region'] + ') — ' + slide10['appeal_type']
slide10 = slide10.sort_values(['date_of_appeal_request_from_ns', 'country']).reset_index(drop=True)
row_order = slide10['row_label'].tolist()


def duration_label(days: int) -> str:
    if days <= 0:
        return 'same-day approval'
    if days == 1:
        return '1 day to approval'
    return f'{days} days to approval'


pre_request_x, pre_request_y = [], []
for row in slide10.itertuples(index=False):
    pre_request_x.extend([crisis_start, row.date_of_appeal_request_from_ns, None])
    pre_request_y.extend([row.row_label, row.row_label, None])

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=pre_request_x,
        y=pre_request_y,
        mode='lines',
        line={'color': '#D6DBE3', 'width': 4},
        hoverinfo='skip',
        showlegend=False,
    )
)
fig.add_trace(
    go.Scatter(
        x=slide10['date_of_appeal_request_from_ns'],
        y=slide10['row_label'],
        mode='markers',
        marker={
            'symbol': 'circle-open',
            'size': 12,
            'line': {'color': '#8A94A6', 'width': 2},
            'color': 'white',
        },
        hovertemplate='Request sent<br>%{y}<br>%{x|%d %b %Y}<extra></extra>',
        showlegend=False,
    )
)

for appeal_type, color in [('EA', IFRC_RED), ('i-DREF', NAVY)]:
    subset = slide10[slide10['appeal_type'] == appeal_type]
    if subset.empty:
        continue

    line_x, line_y = [], []
    for row in subset.itertuples(index=False):
        line_x.extend([row.date_of_appeal_request_from_ns, row.approval_display, None])
        line_y.extend([row.row_label, row.row_label, None])

    fig.add_trace(
        go.Scatter(
            x=line_x,
            y=line_y,
            mode='lines',
            line={'color': color, 'width': 8},
            hoverinfo='skip',
            showlegend=False,
        )
    )
    fig.add_trace(
        go.Scatter(
            x=subset['approval_display'],
            y=subset['row_label'],
            mode='markers',
            marker={
                'symbol': 'circle',
                'size': 14,
                'color': color,
                'line': {'color': FLR_PAPER, 'width': 1.5},
            },
            hovertemplate=f'{appeal_type} approved<br>%{{y}}<br>%{{x|%d %b %Y}}<extra></extra>',
            showlegend=False,
        )
    )

for row in slide10.itertuples(index=False):
    fig.add_annotation(
        x=row.date_of_appeal_request_from_ns,
        y=row.row_label,
        text=f'+{row.request_lag_days}d',
        showarrow=False,
        yshift=18,
        font={'size': 11, 'color': '#8A94A6', 'family': FLR_FONT},
    )
    fig.add_annotation(
        x=row.approval_display + pd.Timedelta(hours=18),
        y=row.row_label,
        text=f'<b>{human_chf(row.total_approved_chf)}</b> │ {duration_label(row.duration_days)}',
        showarrow=False,
        xanchor='left',
        font={'size': 13, 'color': INK, 'family': FLR_FONT},
    )

fig.add_vline(x=crisis_start, line_width=2, line_dash='dash', line_color=INK)
fig.add_annotation(
    x=crisis_start,
    y=1.05,
    xref='x',
    yref='paper',
    text='Crisis start: 28 Feb',
    showarrow=False,
    font={'size': 14, 'color': INK, 'family': FLR_FONT},
)
fig.add_annotation(
    x=0.5,
    y=-0.12,
    xref='paper',
    yref='paper',
    text='Two EA loans were requested at crisis onset, followed by a second wave of i-DREF approvals completed within 0–3 days',
    showarrow=False,
    font={'size': 14, 'color': '#666666', 'family': FLR_FONT},
)

fig.update_layout(
    **flr_layout(
        'Cross-Regional Response Sequence',
        subtitle='Each row shows time from the 28 Feb crisis onset to request, then request to approval for one EA or i-DREF case',
        height=760,
        margin_l=250,
        margin_r=270,
        margin_t=160,
        margin_b=120,
    ),
    xaxis={
        **flr_xaxis('', type='date', dtick=2 * 24 * 60 * 60 * 1000, tickformat='%b %d'),
        'range': [crisis_start - pd.Timedelta(days=1), slide10['approval_display'].max() + pd.Timedelta(days=3)],
    },
    yaxis={
        **flr_yaxis('', showgrid=False, categoryorder='array', categoryarray=row_order[::-1], tickfont={'size': 15, 'color': INK, 'family': FLR_FONT}),
    },
)

save_figure(fig, 'slide_10_response_timeline', width=1500, height=760)
fig